In [2]:
# ============================================================
# Cell 1 - Library dan konfigurasi utama
# ============================================================

from pathlib import Path
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

warnings.filterwarnings("ignore")


# ============================================================
# INPUT / OUTPUT ROOT
# ============================================================

SAMPLED_ROOT = Path("/media/spell/Spell-lab/Lidar/E.Sampled Dataset")
NPZ_OUT_ROOT = Path("/media/spell/Spell-lab/Lidar/G.NPZ Dataset")


# ============================================================
# DATASET CONFIG
# ============================================================

ACTIVITIES = ["Bungkuk", "Duduk", "Jongkok", "Jatuh"]

DEV_SUBJECTS = [
    "Adelia", "Afi", "Aswangga", "Bustan",
    "Dilia", "Eldivo", "Fathir", "Lina",
    "Manda", "Miftah", "Teguh", "Tsamara",
]

TEST_ROOMS = ["Controlled Room", "Uncontrolled Room"]

TEST_SUBJECTS = [
    "Kanaya", "Naila", "Nana", "Rega", "Zaira",
]

FILE_IDS = list(range(1, 10))


# ============================================================
# NPZ CONFIG
# ============================================================

# Semua folder Nxxxx yang tersedia akan diproses otomatis.
# Jika ingin spesifik saja, isi manual misalnya:
# N_FOLDERS_TO_PROCESS = ["N0256", "N0369", "N0384", "N0512"]
N_FOLDERS_TO_PROCESS = None

FRAME_ID_COLUMN = "frame_id"

NPZ_COLUMNS = [
    "Timestamp",
    "X",
    "Y",
    "Z",
    "X_corr",
    "Y_corr",
    "Z_corr",
    "Z_level",
    "Reflectivity",
]

REQUIRED_COLUMNS = [FRAME_ID_COLUMN] + NPZ_COLUMNS

OVERWRITE_NPZ = True
FLOAT_DTYPE = np.float32


print("===== CSV TO NPZ CONVERTER CONFIG =====")
print(f"Input sampled root : {SAMPLED_ROOT}")
print(f"Output NPZ root    : {NPZ_OUT_ROOT}")
print(f"Overwrite NPZ      : {OVERWRITE_NPZ}")
print(f"NPZ columns        : {NPZ_COLUMNS}")

===== CSV TO NPZ CONVERTER CONFIG =====
Input sampled root : /media/spell/Spell-lab/Lidar/E.Sampled Dataset
Output NPZ root    : /media/spell/Spell-lab/Lidar/G.NPZ Dataset
Overwrite NPZ      : True
NPZ columns        : ['Timestamp', 'X', 'Y', 'Z', 'X_corr', 'Y_corr', 'Z_corr', 'Z_level', 'Reflectivity']


In [3]:
# ============================================================
# Cell 2 - Buat output root dan deteksi folder N target
# ============================================================

NPZ_OUT_ROOT.mkdir(parents=True, exist_ok=True)

if N_FOLDERS_TO_PROCESS is None:
    n_folders = sorted([
        p.name for p in SAMPLED_ROOT.iterdir()
        if p.is_dir() and p.name.startswith("N")
    ])
else:
    n_folders = N_FOLDERS_TO_PROCESS

if len(n_folders) == 0:
    raise FileNotFoundError(
        f"Tidak ada folder Nxxxx ditemukan di: {SAMPLED_ROOT}"
    )

print("===== N FOLDERS TO PROCESS =====")
for n_folder in n_folders:
    print("-", n_folder)

config = {
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "sampled_root": str(SAMPLED_ROOT),
    "npz_out_root": str(NPZ_OUT_ROOT),
    "n_folders": n_folders,
    "frame_id_column": FRAME_ID_COLUMN,
    "npz_columns": NPZ_COLUMNS,
    "required_columns": REQUIRED_COLUMNS,
    "overwrite_npz": OVERWRITE_NPZ,
    "float_dtype": str(FLOAT_DTYPE),
}

config_path = NPZ_OUT_ROOT / "csv_to_npz_converter_config.json"

with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print(f"\nConverter config saved to: {config_path}")

===== N FOLDERS TO PROCESS =====
- N0256
- N0369
- N0384
- N0512
- N1024

Converter config saved to: /media/spell/Spell-lab/Lidar/G.NPZ Dataset/csv_to_npz_converter_config.json


In [4]:
# ============================================================
# Cell 3 - Build manifest seluruh CSV sampled
# ============================================================

def build_csv_to_npz_manifest():
    records = []

    for n_folder in n_folders:
        n_input_root = SAMPLED_ROOT / n_folder
        n_output_root = NPZ_OUT_ROOT / n_folder

        # Dataset Development
        for activity in ACTIVITIES:
            for subject in DEV_SUBJECTS:
                for file_id in FILE_IDS:
                    csv_name = f"{file_id}.csv"
                    npz_name = f"{file_id}.npz"

                    input_csv = (
                        n_input_root
                        / "Dataset Development"
                        / activity
                        / subject
                        / csv_name
                    )

                    output_npz = (
                        n_output_root
                        / "Dataset Development"
                        / activity
                        / subject
                        / npz_name
                    )

                    records.append({
                        "n_folder": n_folder,
                        "dataset": "development",
                        "room": "",
                        "activity": activity,
                        "subject": subject,
                        "file_id": file_id,
                        "input_csv": input_csv,
                        "output_npz": output_npz,
                        "exists": input_csv.exists(),
                    })

        # Dataset Testing
        for room in TEST_ROOMS:
            for activity in ACTIVITIES:
                for subject in TEST_SUBJECTS:
                    for file_id in FILE_IDS:
                        csv_name = f"{file_id}.csv"
                        npz_name = f"{file_id}.npz"

                        input_csv = (
                            n_input_root
                            / "Dataset Testing"
                            / room
                            / activity
                            / subject
                            / csv_name
                        )

                        output_npz = (
                            n_output_root
                            / "Dataset Testing"
                            / room
                            / activity
                            / subject
                            / npz_name
                        )

                        records.append({
                            "n_folder": n_folder,
                            "dataset": "testing",
                            "room": room,
                            "activity": activity,
                            "subject": subject,
                            "file_id": file_id,
                            "input_csv": input_csv,
                            "output_npz": output_npz,
                            "exists": input_csv.exists(),
                        })

    return pd.DataFrame(records)


manifest_df = build_csv_to_npz_manifest()

manifest_save = manifest_df.copy()
manifest_save["input_csv"] = manifest_save["input_csv"].astype(str)
manifest_save["output_npz"] = manifest_save["output_npz"].astype(str)

manifest_path = NPZ_OUT_ROOT / "csv_to_npz_manifest.csv"
manifest_save.to_csv(manifest_path, index=False)

missing_df = manifest_df[~manifest_df["exists"]].copy()
missing_save = missing_df.copy()
missing_save["input_csv"] = missing_save["input_csv"].astype(str)
missing_save["output_npz"] = missing_save["output_npz"].astype(str)

missing_path = NPZ_OUT_ROOT / "missing_sampled_csv_files.csv"
missing_save.to_csv(missing_path, index=False)

existing_df = manifest_df[manifest_df["exists"]].copy()

print("===== MANIFEST SUMMARY =====")
print(f"Total expected CSV : {len(manifest_df)}")
print(f"Existing CSV       : {len(existing_df)}")
print(f"Missing CSV        : {len(missing_df)}")
print(f"Manifest saved     : {manifest_path}")
print(f"Missing saved      : {missing_path}")

if len(existing_df) == 0:
    raise FileNotFoundError("Tidak ada CSV sampled yang ditemukan.")

===== MANIFEST SUMMARY =====
Total expected CSV : 3960
Existing CSV       : 3960
Missing CSV        : 0
Manifest saved     : /media/spell/Spell-lab/Lidar/G.NPZ Dataset/csv_to_npz_manifest.csv
Missing saved      : /media/spell/Spell-lab/Lidar/G.NPZ Dataset/missing_sampled_csv_files.csv


In [5]:
# ============================================================
# Cell 4 - Helper load CSV dan validasi
# ============================================================

def load_and_validate_sampled_csv(csv_path):
    df = pd.read_csv(csv_path)

    missing_cols = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns: {missing_cols}")

    df = df[REQUIRED_COLUMNS].copy()

    for col in REQUIRED_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=REQUIRED_COLUMNS).copy()

    df[FRAME_ID_COLUMN] = df[FRAME_ID_COLUMN].astype(int)

    for col in NPZ_COLUMNS:
        df[col] = df[col].astype(FLOAT_DTYPE)

    df = df.sort_values([FRAME_ID_COLUMN]).reset_index(drop=True)

    return df

In [6]:
# ============================================================
# Cell 5 - Helper convert satu CSV ke NPZ
# ============================================================

def convert_one_csv_to_npz(row):
    input_csv = Path(row["input_csv"])
    output_npz = Path(row["output_npz"])

    info = {
        "n_folder": row["n_folder"],
        "dataset": row["dataset"],
        "room": row["room"],
        "activity": row["activity"],
        "subject": row["subject"],
        "file_id": int(row["file_id"]),
        "input_csv": str(input_csv),
        "output_npz": str(output_npz),
    }

    if not input_csv.exists():
        return {
            **info,
            "status": "missing_input",
            "n_rows_csv": 0,
            "n_frames": 0,
            "n_points_per_frame": np.nan,
        }

    if output_npz.exists() and not OVERWRITE_NPZ:
        return {
            **info,
            "status": "skipped_output_exists",
            "n_rows_csv": np.nan,
            "n_frames": np.nan,
            "n_points_per_frame": np.nan,
        }

    try:
        df = load_and_validate_sampled_csv(input_csv)
    except Exception as e:
        return {
            **info,
            "status": f"read_or_validation_error: {e}",
            "n_rows_csv": 0,
            "n_frames": 0,
            "n_points_per_frame": np.nan,
        }

    if len(df) == 0:
        return {
            **info,
            "status": "empty_after_validation",
            "n_rows_csv": 0,
            "n_frames": 0,
            "n_points_per_frame": np.nan,
        }

    frame_counts = df.groupby(FRAME_ID_COLUMN).size()
    unique_counts = sorted(frame_counts.unique().tolist())

    if len(unique_counts) != 1:
        return {
            **info,
            "status": f"invalid_variable_points_per_frame: {unique_counts[:10]}",
            "n_rows_csv": len(df),
            "n_frames": int(frame_counts.shape[0]),
            "n_points_per_frame": np.nan,
        }

    n_points_per_frame = int(unique_counts[0])
    frame_ids = frame_counts.index.to_numpy(dtype=np.int32)
    n_frames = len(frame_ids)

    data_frames = []

    for frame_id in frame_ids:
        frame_df = df[df[FRAME_ID_COLUMN] == frame_id]
        arr = frame_df[NPZ_COLUMNS].to_numpy(dtype=FLOAT_DTYPE)

        if arr.shape != (n_points_per_frame, len(NPZ_COLUMNS)):
            return {
                **info,
                "status": f"invalid_frame_shape_frame_{frame_id}",
                "n_rows_csv": len(df),
                "n_frames": n_frames,
                "n_points_per_frame": n_points_per_frame,
            }

        data_frames.append(arr)

    data = np.stack(data_frames, axis=0).astype(FLOAT_DTYPE)

    output_npz.parent.mkdir(parents=True, exist_ok=True)

    np.savez_compressed(
        output_npz,
        data=data,
        frame_ids=frame_ids,
        feature_names=np.array(NPZ_COLUMNS, dtype=object),
        dataset=np.array(row["dataset"], dtype=object),
        room=np.array(row["room"], dtype=object),
        activity=np.array(row["activity"], dtype=object),
        subject=np.array(row["subject"], dtype=object),
        file_id=np.array(int(row["file_id"]), dtype=np.int32),
        source_csv=np.array(str(input_csv), dtype=object),
        n_folder=np.array(row["n_folder"], dtype=object),
        n_points_per_frame=np.array(n_points_per_frame, dtype=np.int32),
    )

    return {
        **info,
        "status": "converted",
        "n_rows_csv": int(len(df)),
        "n_frames": int(n_frames),
        "n_points_per_frame": int(n_points_per_frame),
        "data_shape": str(data.shape),
    }

In [7]:
# ============================================================
# Cell 6 - Mini test satu file
# ============================================================

test_row = existing_df.iloc[0].to_dict()

print("===== MINI TEST CSV TO NPZ =====")
print("Input CSV :", test_row["input_csv"])
print("Output NPZ:", test_row["output_npz"])

mini_result = convert_one_csv_to_npz(test_row)

print("\nMini result:")
for k, v in mini_result.items():
    print(f"{k}: {v}")

mini_npz_path = Path(mini_result["output_npz"])

if mini_npz_path.exists():
    loaded = np.load(mini_npz_path, allow_pickle=True)

    print("\n===== MINI NPZ CONTENT =====")
    print("Keys:", loaded.files)
    print("data shape:", loaded["data"].shape)
    print("frame_ids shape:", loaded["frame_ids"].shape)
    print("feature_names:", loaded["feature_names"])
    print("activity:", loaded["activity"].item())
    print("subject:", loaded["subject"].item())
    print("room:", loaded["room"].item())
    print("file_id:", loaded["file_id"].item())
    print("source_csv:", loaded["source_csv"].item())

===== MINI TEST CSV TO NPZ =====
Input CSV : /media/spell/Spell-lab/Lidar/E.Sampled Dataset/N0256/Dataset Development/Bungkuk/Adelia/1.csv
Output NPZ: /media/spell/Spell-lab/Lidar/G.NPZ Dataset/N0256/Dataset Development/Bungkuk/Adelia/1.npz

Mini result:
n_folder: N0256
dataset: development
room: 
activity: Bungkuk
subject: Adelia
file_id: 1
input_csv: /media/spell/Spell-lab/Lidar/E.Sampled Dataset/N0256/Dataset Development/Bungkuk/Adelia/1.csv
output_npz: /media/spell/Spell-lab/Lidar/G.NPZ Dataset/N0256/Dataset Development/Bungkuk/Adelia/1.npz
status: converted
n_rows_csv: 14080
n_frames: 55
n_points_per_frame: 256
data_shape: (55, 256, 9)

===== MINI NPZ CONTENT =====
Keys: ['data', 'frame_ids', 'feature_names', 'dataset', 'room', 'activity', 'subject', 'file_id', 'source_csv', 'n_folder', 'n_points_per_frame']
data shape: (55, 256, 9)
frame_ids shape: (55,)
feature_names: ['Timestamp' 'X' 'Y' 'Z' 'X_corr' 'Y_corr' 'Z_corr' 'Z_level'
 'Reflectivity']
activity: Bungkuk
subject: Adelia

In [8]:
# ============================================================
# Cell 7 - Convert seluruh CSV sampled ke NPZ
# ============================================================

conversion_results = []

for _, row in tqdm(existing_df.iterrows(), total=len(existing_df), desc="Converting CSV to NPZ"):
    result = convert_one_csv_to_npz(row)
    conversion_results.append(result)

conversion_summary_df = pd.DataFrame(conversion_results)

conversion_summary_path = NPZ_OUT_ROOT / "csv_to_npz_conversion_summary.csv"
conversion_summary_df.to_csv(conversion_summary_path, index=False)

print("===== CONVERSION DONE =====")
print(f"Total processed CSV : {len(conversion_summary_df)}")
print(f"Summary saved       : {conversion_summary_path}")

display(conversion_summary_df["status"].value_counts(dropna=False).to_frame("count"))

Converting CSV to NPZ:   0%|          | 0/3960 [00:00<?, ?it/s]

===== CONVERSION DONE =====
Total processed CSV : 3960
Summary saved       : /media/spell/Spell-lab/Lidar/G.NPZ Dataset/csv_to_npz_conversion_summary.csv


,count
status,
converted,3960


In [9]:
# ============================================================
# Cell 8 - Global sanity check NPZ
# ============================================================

converted_df = conversion_summary_df[
    conversion_summary_df["status"] == "converted"
].copy()

print("===== GLOBAL NPZ SANITY CHECK =====")
print(f"Converted files: {len(converted_df)}")

if len(converted_df) > 0:
    print("\nN folder distribution:")
    display(converted_df["n_folder"].value_counts().sort_index().to_frame("count"))

    print("\nDataset distribution:")
    display(converted_df["dataset"].value_counts().to_frame("count"))

    print("\nRoom distribution:")
    display(converted_df["room"].value_counts(dropna=False).to_frame("count"))

    print("\nPoints per frame distribution:")
    display(converted_df["n_points_per_frame"].value_counts().sort_index().to_frame("file_count"))

    print("\nFrame count statistics:")
    display(converted_df["n_frames"].describe().to_frame("n_frames"))

===== GLOBAL NPZ SANITY CHECK =====
Converted files: 3960

N folder distribution:


,count
n_folder,
N0256,792
N0369,792
N0384,792
N0512,792
N1024,792



Dataset distribution:


,count
dataset,
development,2160
testing,1800



Room distribution:


,count
room,
,2160
Controlled Room,900
Uncontrolled Room,900



Points per frame distribution:


,file_count
n_points_per_frame,
256,792
369,792
384,792
512,792
1024,792



Frame count statistics:


,n_frames
count,3960.000000
mean,59.811869
std,7.439829
min,35.000000
25%,56.000000
50%,58.000000
75%,64.000000
max,113.000000


In [10]:
# ============================================================
# Cell 9 - Verifikasi random beberapa NPZ
# ============================================================

N_RANDOM_CHECK = min(5, len(converted_df))

if N_RANDOM_CHECK > 0:
    sample_check_df = converted_df.sample(N_RANDOM_CHECK, random_state=42)

    for _, row in sample_check_df.iterrows():
        path = Path(row["output_npz"])
        loaded = np.load(path, allow_pickle=True)

        data = loaded["data"]
        frame_ids = loaded["frame_ids"]
        feature_names = loaded["feature_names"]

        print("\n===== NPZ CHECK =====")
        print("Path:", path)
        print("data shape:", data.shape)
        print("frame_ids shape:", frame_ids.shape)
        print("feature_names:", feature_names)
        print("activity:", loaded["activity"].item())
        print("subject:", loaded["subject"].item())
        print("room:", loaded["room"].item())
        print("file_id:", loaded["file_id"].item())

        assert data.ndim == 3
        assert frame_ids.ndim == 1
        assert data.shape[0] == frame_ids.shape[0]
        assert data.shape[2] == len(NPZ_COLUMNS)
        assert list(feature_names) == NPZ_COLUMNS

    print("\nRandom NPZ verification: PASSED")
else:
    print("No converted NPZ files to verify.")


===== NPZ CHECK =====
Path: /media/spell/Spell-lab/Lidar/G.NPZ Dataset/N0256/Dataset Development/Duduk/Dilia/6.npz
data shape: (55, 256, 9)
frame_ids shape: (55,)
feature_names: ['Timestamp' 'X' 'Y' 'Z' 'X_corr' 'Y_corr' 'Z_corr' 'Z_level'
 'Reflectivity']
activity: Duduk
subject: Dilia
room: 
file_id: 6

===== NPZ CHECK =====
Path: /media/spell/Spell-lab/Lidar/G.NPZ Dataset/N0369/Dataset Development/Jongkok/Afi/9.npz
data shape: (59, 369, 9)
frame_ids shape: (59,)
feature_names: ['Timestamp' 'X' 'Y' 'Z' 'X_corr' 'Y_corr' 'Z_corr' 'Z_level'
 'Reflectivity']
activity: Jongkok
subject: Afi
room: 
file_id: 9

===== NPZ CHECK =====
Path: /media/spell/Spell-lab/Lidar/G.NPZ Dataset/N0384/Dataset Development/Jongkok/Eldivo/2.npz
data shape: (64, 384, 9)
frame_ids shape: (64,)
feature_names: ['Timestamp' 'X' 'Y' 'Z' 'X_corr' 'Y_corr' 'Z_corr' 'Z_level'
 'Reflectivity']
activity: Jongkok
subject: Eldivo
room: 
file_id: 2

===== NPZ CHECK =====
Path: /media/spell/Spell-lab/Lidar/G.NPZ Dataset/N

In [11]:
# ============================================================
# Cell 10 - Final artifact check
# ============================================================

print("===== FINAL ARTIFACT CHECK =====")
print("NPZ output root:", NPZ_OUT_ROOT)
print("Config exists:", config_path.exists())
print("Manifest exists:", manifest_path.exists())
print("Missing report exists:", missing_path.exists())
print("Conversion summary exists:", conversion_summary_path.exists())

print("\nFolder structure preview:")
for n_folder in n_folders:
    p = NPZ_OUT_ROOT / n_folder
    print(f"{n_folder}: exists={p.exists()} | {p}")

print("\nStatus summary:")
display(conversion_summary_df["status"].value_counts(dropna=False).to_frame("count"))

===== FINAL ARTIFACT CHECK =====
NPZ output root: /media/spell/Spell-lab/Lidar/G.NPZ Dataset
Config exists: True
Manifest exists: True
Missing report exists: True
Conversion summary exists: True

Folder structure preview:
N0256: exists=True | /media/spell/Spell-lab/Lidar/G.NPZ Dataset/N0256
N0369: exists=True | /media/spell/Spell-lab/Lidar/G.NPZ Dataset/N0369
N0384: exists=True | /media/spell/Spell-lab/Lidar/G.NPZ Dataset/N0384
N0512: exists=True | /media/spell/Spell-lab/Lidar/G.NPZ Dataset/N0512
N1024: exists=True | /media/spell/Spell-lab/Lidar/G.NPZ Dataset/N1024

Status summary:


,count
status,
converted,3960
